# Graph topology: from four dummy electrodes to our EEG dataset

Run the cells from top to bottom. We first build a tiny example, then use the real
`graphdataversiontwo` dataset and `ourexperimentversionfive/data` loader. This notebook
reads saved data; it does not regenerate features or train the GNN.

**Topology means which nodes are connected.** Node features describe an electrode;
edge weights describe the strength assigned to a connection. Electrode positions
are geometry, not automatically topology. An edge is a computational connection,
not proof of a physical or causal connection in the brain.

Our saved graph contains every electrode pair. Its **unweighted topology is the same
for every trial**. Node features and connectivity weights change between trials.
People sometimes use “weighted topology” to include those weight patterns; here we
name topology and weights separately to avoid ambiguity.

Requirements: the project's NumPy, pandas, matplotlib, PyTorch, PyG,
mne-connectivity and scikit-learn environment, with a Jupyter kernel.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch_geometric
from IPython.display import display
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/datautils/graphdataversiontwo').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the EEG repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(7)
print('Repository:', ROOT)
print('PyTorch:', torch.__version__, '| PyG:', torch_geometric.__version__)


## 1. Four dummy electrodes, two features, one trial label

These numbers are invented, not computed EEG measurements. Row `i` of `x` always
belongs to electrode `i`. We use two power features only to keep the table readable.
`y` labels the **whole trial**, not individual electrodes.


In [ ]:
names = ['A', 'B', 'C', 'D']
x = torch.tensor([[1., 2.], [3., 1.], [2., 4.], [4., 3.]], dtype=torch.float32)
y = torch.tensor([0], dtype=torch.long)  # invented trial class
# Three undirected connections: A--B, B--C, C--D.
# Store both directions so messages can travel both ways.
edge_index = torch.tensor([[0, 1, 2, 1, 2, 3],
                           [1, 2, 3, 0, 1, 2]], dtype=torch.long)
edge_weight = torch.tensor([0.2, 0.8, 0.4, 0.2, 0.8, 0.4])
graph = Data(x=x, edge_index=edge_index, edge_attr=edge_weight[:, None], y=y)
assert graph.validate(raise_on_error=True)
display(pd.DataFrame(x.numpy(), index=names, columns=['alpha_power', 'beta_power']))
display(pd.DataFrame({'edge position k': range(graph.num_edges),
                      'source': [names[i] for i in edge_index[0]],
                      'target': [names[i] for i in edge_index[1]],
                      'weight': edge_weight.numpy()}))
print(graph)


## 2. See topology and weights separately

The first two pictures have identical connections but different weights. The third
adds A--D, so its topology changes. Line thickness shows weight; coordinates are
just a drawing layout. A complete four-node graph would have six unique pairs,
or 12 directed entries. Our chain has three unique pairs, or six entries.


In [ ]:
positions = np.array([[0, 1], [1, 1], [1, 0], [0, 0]])
def draw_graph(ax, pairs, weights, title):
    for (a, b), w in zip(pairs, weights):
        xy = positions[[a, b]]
        ax.plot(xy[:, 0], xy[:, 1], color='steelblue', linewidth=1 + 5*w)
        mid = xy.mean(axis=0)
        ax.text(*mid, f'{w:.1f}', backgroundcolor='white', ha='center')
    ax.scatter(*positions.T, s=650, color='lightyellow', edgecolors='black', zorder=3)
    for name, xy in zip(names, positions):
        ax.text(*xy, name, ha='center', va='center', zorder=4)
    ax.set(xlim=(-.3, 1.3), ylim=(-.3, 1.3), title=title, aspect='equal')
    ax.axis('off')
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
pairs = [(0, 1), (1, 2), (2, 3)]
draw_graph(axes[0], pairs, [.2, .8, .4], 'Original topology and weights')
draw_graph(axes[1], pairs, [.8, .2, .6], 'Same topology, different weights')
draw_graph(axes[2], pairs + [(0, 3)], [.2, .8, .4, .5], 'Different topology: add A—D')
plt.show()


## 3. What PyTorch Geometric needs

| Tensor | Dummy shape | One real trial | Meaning |
|---|---|---|---|
| `x` (floating point) | `(4, 2)` | `(29, 8)` | Rows are nodes; columns are features |
| `edge_index` (`torch.long`) | `(2, 6)` | `(2, 812)` | Column k is source → target |
| `edge_attr` (floating point) | `(6, 1)` | `(812, 1)` | Loader stores alpha wPLI in column 0 |
| `edge_weight` (floating point) | `(6,)` | `(812,)` | Scalar weights explicitly passed to GCNConv |
| `y` (`torch.long`) | `(1,)` | `(1,)` | Trial class |
| `batch` (`torch.long`) | One entry per node | `(B × 29,)` | Which graph owns each node in a batch |

`Data` is a container. Storing `edge_attr` does not automatically make a layer use it.
Our engine explicitly passes `edge_attr.squeeze(-1)` to the model, which passes it
to `GCNConv`. Five-band `(812, 5)` attributes cannot simply be supplied as scalar
GCN weights; the current loader selects alpha first. `pos` is optional, and the
current GCN does not consume electrode coordinates.

PyG's `Data.validate()` checks basic representation validity, not our complete-pair,
symmetry, wPLI-range, or electrode-name contract.

References: [GCNConv](https://pytorch-geometric.readthedocs.io/en/stable/generated/torch_geometric.nn.conv.GCNConv.html),
[Data](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.Data.html),
[batching](https://pytorch-geometric.readthedocs.io/en/latest/advanced/batching.html).


## 4. A small message-passing calculation

For an easy first calculation, sum each source feature into its target, multiplied
by its edge weight. For B, the result is `0.2*x[A] + 0.8*x[C] = [1.8, 3.6]`.
This illustrative sum omits GCN's learned transform, degree normalization and
self-loops. The second calculation uses the actual GCN layer.

GCNConv normally adds self-loops during its forward calculation when normalization
is enabled. Therefore “no self-loops in the saved edge list” does **not** mean a node
cannot use its own features in the model. Keep `cached=False` for changing trial weights.


In [ ]:
source, target = graph.edge_index
messages = graph.x[source] * edge_weight[:, None]
summed = torch.zeros_like(graph.x)
summed.index_add_(0, target, messages)
assert torch.allclose(summed[1], torch.tensor([1.8, 3.6]))
display(pd.DataFrame(summed.numpy(), index=names, columns=['received feature 1', 'received feature 2']))
conv = GCNConv(2, 2, bias=False, cached=False)
with torch.no_grad():
    conv.lin.weight.copy_(torch.eye(2))  # identity transform to simplify inspection
    weighted = conv(graph.x, graph.edge_index, graph.edge_attr.squeeze(-1))
    unweighted = conv(graph.x, graph.edge_index)  # all provided connections weight 1
print('Actual normalized GCN output:\n', weighted)
print('Maximum change when weights are omitted:', (weighted-unweighted).abs().max().item())


## 5. Batching keeps trials disconnected

PyG concatenates nodes and offsets the second graph's electrode indices by four.
It does not connect trial 1 to trial 2. `batch` records node membership and `ptr`
records graph boundaries. The real loader performs this operation for 29-node trials.


In [ ]:
graph2 = graph.clone()
graph2.x = graph2.x + 0.5
graph2.y = torch.tensor([1])
batched = Batch.from_data_list([graph, graph2])
print(batched)
print('edge_index:\n', batched.edge_index)
print('batch:', batched.batch.tolist(), '| ptr:', batched.ptr.tolist())
assert torch.equal(batched.edge_index[:, graph.num_edges:], graph.edge_index + 4)
assert torch.equal(batched.batch[batched.edge_index[0]], batched.batch[batched.edge_index[1]])


## 6. Check the production matrix-to-edge conversion using known answers

Use the actual generator functions, with a symmetric dummy matrix whose every
pair/band has a distinct known value. This catches pair and band ordering mistakes
in extraction. It does not test the EEG estimator or file-writing pipeline.
A symmetric matrix equals its transpose, so pure transposition is undetectable
and harmless here; confusing channels, bands, or trials is a different issue.


In [ ]:
from src.datautils.graphdataversiontwo.edge_features import build_edge_index, build_edge_features
sentinel = np.zeros((5, 4, 4))  # band, source, target
expected_by_pair = {}
for band in range(5):
    pair_number = 0
    for a in range(4):
        for b in range(a + 1, 4):
            value = (band * 6 + pair_number + 1) / 100
            sentinel[band, a, b] = sentinel[band, b, a] = value
            expected_by_pair[band, a, b] = value
            pair_number += 1
sentinel_index = build_edge_index(4)
extracted = build_edge_features({'wpli': sentinel}, sentinel_index)['wpli']
for k, (a, b) in enumerate(sentinel_index.T):
    for band in range(5):
        assert extracted[k, band] == expected_by_pair[band, min(a, b), max(a, b)]
print('PASS: all 12 directed pairs × 5 bands match independently assigned values.')


## 7. Load version-two data and run the full validator

The saved-data validator checks every trial and all node/edge variants, including pair ordering, ranges, and trial alignment. The experiment adapter additionally checks its source contract and selects all eight columns from the eight saved features. We run both checks below.


In [ ]:
from src.datautils.graphdataversiontwo.config import OUTPUT_ROOT, CHANNEL_NAMES
from src.datautils.graphdataversiontwo.saved_dataset import load_dataset
from src.datautils.graphdataversiontwo.validation import validate_saved_dataset
from src.ourexperimentversionfive.data.validation import validate_dataset, alpha_band_index
DATASET_PATH = OUTPUT_ROOT  # Change if your saved dataset lives elsewhere.
if not DATASET_PATH.is_dir():
    raise FileNotFoundError(f'Saved dataset missing: {DATASET_PATH}. Set DATASET_PATH above.')
dataset = load_dataset(DATASET_PATH)
report = validate_saved_dataset(DATASET_PATH, expected_trials_per_subject=40)
validate_dataset(dataset)
assert list(dataset.metadata['channel_names']) == list(CHANNEL_NAMES)
print('PASS: full saved-data validator and experiment contract.')
print({k: report[k] for k in ['n_samples', 'n_subjects', 'class_counts']})
print('Node features:', dataset.metadata['node_feature_names'])
print('Bands:', dataset.metadata['band_names'])
print('Nodes:', dataset.nodes['without_csd'].shape)
print('Raw wPLI:', dataset.edges['wpli_without_csd'].shape)


## 8. Make the topology checks visible

A complete graph has degree 28 at each electrode, 406 unique undirected pairs,
and 812 directed entries. Reverse directions are intentional, not duplicate
ordered pairs. Exact edge ordering is a saved-file contract; PyG itself accepts
other orderings if edge values are reordered together.


In [ ]:
ei = np.asarray(dataset.edge_index)
channels = dataset.metadata['channel_names']
n = len(channels)
actual_pairs = [tuple(pair) for pair in ei.T]
expected_pairs = {(a, b) for a in range(n) for b in range(n) if a != b}
checks = {
    'integer indices': np.issubdtype(ei.dtype, np.integer),
    'shape (2, 812)': ei.shape == (2, n*(n-1)),
    'indices in range': bool(((ei >= 0) & (ei < n)).all()),
    'no self-loops': bool((ei[0] != ei[1]).all()),
    'no duplicate ordered pairs': len(set(actual_pairs)) == len(actual_pairs),
    'no missing or unexpected pairs': set(actual_pairs) == expected_pairs,
    'canonical generator ordering': np.array_equal(ei, build_edge_index(n)),
}
display(pd.Series(checks, name='passed').to_frame())
assert all(checks.values())
display(pd.DataFrame({'electrode': channels,
                      'out_degree': np.bincount(ei[0], minlength=n),
                      'in_degree': np.bincount(ei[1], minlength=n)}))


## 9. Check every raw wPLI value and inspect named pairs

The report below covers **all subjects, trials and bands**, before normalization
or alpha selection. Range tolerance is `1e-6`, matching the full validator;
we also show strict outside-[0,1] counts so tolerated roundoff remains visible.
The full validator above already verifies reverse values using its tighter
absolute/relative tolerance. Below we display the maximum discrepancy as well.


In [ ]:
wpli = dataset.edges['wpli_without_csd']
lookup = {pair: k for k, pair in enumerate(actual_pairs)}
reverse = np.array([lookup[b, a] for a, b in actual_pairs])
rows = []
for b, band in enumerate(dataset.metadata['band_names']):
    values = np.asarray(wpli[:, :, b])
    bad = ~np.isfinite(values) | (values < -1e-6) | (values > 1+1e-6)
    if bad.any():
        trial, edge = np.argwhere(bad)[0]
        raise ValueError(f'{dataset.samples[trial]}, band={band}, pair={actual_pairs[edge]}, value={values[trial, edge]}')
    rows.append({'band': band, 'values_checked': values.size,
                 'min': values.min(), 'max': values.max(),
                 'strict_outside_count': int(((values < 0) | (values > 1)).sum()),
                 'max_reverse_difference': np.abs(values-values[:, reverse]).max()})
display(pd.DataFrame(rows))
alpha = alpha_band_index(dataset)
trial = 0
example_edges = [0, 1, 405, 406, 811]
display(pd.DataFrame([{'edge k': k, 'source': channels[ei[0,k]],
                       'target': channels[ei[1,k]], 'alpha wPLI': float(wpli[trial,k,alpha])}
                      for k in example_edges]))


In [ ]:
# Reconstruct matrices for inspection, not as an independent proof of correctness.
adjacency = np.zeros((n, n))
adjacency[ei[0], ei[1]] = 1
weighted_adjacency = np.zeros((n, n))
weighted_adjacency[ei[0], ei[1]] = wpli[trial, :, alpha]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, matrix, title in zip(axes, [adjacency, weighted_adjacency],
                            ['Shared topology: all off-diagonal pairs', 'Trial 0: alpha wPLI weights']):
    im = ax.imshow(matrix, vmin=0, vmax=1, cmap='viridis')
    ax.set_title(title)
    ax.set_xticks(range(n), channels, rotation=90, fontsize=7)
    ax.set_yticks(range(n), channels, fontsize=7)
    ax.set_xlabel('Target electrode'); ax.set_ylabel('Source electrode')
    fig.colorbar(im, ax=ax, fraction=.046)
plt.tight_layout(); plt.show()


## 10. Verify the version-five loader and one model forward pass

Select `NODE_NORMALIZATION = "none"` or `"zscore"`. Model inputs contain all five band powers, normalized spectral entropy (1–40 Hz), and broadband Hjorth mobility/complexity. Saved arrays retain eight columns. Z-scoring, when enabled, uses only training trials; disabled mode preserves selected values exactly. Alpha wPLI remains a single unchanged scalar per edge.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from src.ourexperimentversionfive.data.without_csd_alpha_wpli import (
    fit_feature_normalization, create_within_subject_dataloaders)
from src.ourexperimentversionfive.data.loso_split import GraphDataLoaderConfig
from src.ourexperimentversionfive.model.gcn import EEGGCN1
from src.ourexperimentversionfive.data.validation import select_node_features, NODE_FEATURE_NAMES
NODE_NORMALIZATION = "none"  # Choose "none" or "zscore".
print("Selected features:", NODE_FEATURE_NAMES, "| normalization:", NODE_NORMALIZATION)
subject_ids = np.array([s['subject'] for s in dataset.samples])
subject = int(subject_ids[0])
indices = np.flatnonzero(subject_ids == subject)
labels = np.asarray(dataset.labels)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_local, test_local = next(cv.split(indices, labels[indices]))
train_indices, test_indices = indices[train_local], indices[test_local]
assert not set(train_indices) & set(test_indices)
normalization = fit_feature_normalization(dataset, graph_indices=train_indices, mode=NODE_NORMALIZATION)
train_loader, evaluation_loader, _ = create_within_subject_dataloaders(
    dataset=dataset, train_graph_indices=train_indices,
    evaluation_graph_indices=test_indices,
    config=GraphDataLoaderConfig(batch_size=4, num_workers=0, node_normalization=NODE_NORMALIZATION),
    dataset_validated=True, normalization=normalization)
if NODE_NORMALIZATION == "zscore":
    mean = normalization.mean_by_subject[subject].numpy()
    std = normalization.standard_deviation_by_subject[subject].numpy()
    training_x = select_node_features(dataset, 'without_csd', train_indices).astype(np.float64)
    np.testing.assert_allclose(mean, training_x.mean(axis=(0, 1)), rtol=1e-6, atol=1e-6)
    np.testing.assert_allclose(std, np.maximum(training_x.std(axis=(0, 1)), 1e-8), rtol=1e-6, atol=1e-6)
else:
    assert normalization.mean_by_subject == {} and normalization.fit_graph_indices == ()
for position, saved_index in enumerate(test_indices):
    loaded = evaluation_loader.dataset[position]
    assert loaded.validate(raise_on_error=True)
    np.testing.assert_array_equal(loaded.edge_index.numpy(), ei)
    np.testing.assert_array_equal(loaded.edge_attr[:, 0].numpy(), wpli[saved_index, :, alpha])
    expected_x = select_node_features(dataset, 'without_csd', saved_index)
    if NODE_NORMALIZATION == 'zscore':
        expected_x = (expected_x - mean) / std
    np.testing.assert_allclose(loaded.x.numpy(), expected_x, rtol=1e-5, atol=1e-6)
    assert loaded.y.item() == labels[saved_index]
real_batch = next(iter(evaluation_loader))
src, dst = real_batch.edge_index
assert torch.equal(real_batch.batch[src], real_batch.batch[dst])
assert real_batch.x.shape == (real_batch.num_graphs * 29, 8)
assert real_batch.edge_attr.shape == (real_batch.num_graphs * 812, 1)
model = EEGGCN1().eval()
with torch.no_grad():
    output = model(real_batch.x, real_batch.edge_index,
                   real_batch.edge_attr.squeeze(-1), real_batch.batch)
assert output.shape == (real_batch.num_graphs, 2) and torch.isfinite(output).all()
print(f'PASS: {len(test_indices)} evaluation graphs aligned; normalization mode {NODE_NORMALIZATION} checked.')
print(real_batch)
print('Untrained model output shape:', tuple(output.shape))


## 11. Optional: connectivity-only versus nodes-only LDA

This is a feature-information check, not a topology test. Enable the next cell to
compare representations on identical five-fold splits. Default scope is one subject;
set `SUBJECTS = sorted(set(subject_ids))` for all 50. No subject is selected by score.

With 40 trials, each fold has only 32 training trials. Use shrinkage LDA for all representations; optional training-fold scaling follows
`NODE_NORMALIZATION`. This classifier comparison is separate from GCN training. The 812 directed columns contain only 406
unique pair measurements when symmetry holds; we include the 406-column version
as a redundancy check. Regularization means duplicate columns need not give exactly
the same predictions. The seven-feature baseline uses the earlier representation,
but does not claim to reproduce its original seed or estimator settings.

A score from one subject or eight test trials per fold is noisy. Above 0.5 alone is
not a significance test. Before scientific conclusions, add label-permutation tests
of the full CV procedure and paired subject-level uncertainty for differences.
Random trial folds test trial generalization within this recording, not unseen-session
or unseen-subject generalization.


In [ ]:
RUN_CLASSIFIERS = False
SUBJECTS = [subject]
if RUN_CLASSIFIERS:
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    from sklearn.metrics import balanced_accuracy_score
    from threadpoolctl import threadpool_limits
    scores = []
    unique_pair_mask = ei[0] < ei[1]
    with threadpool_limits(limits=1):
        for sid in SUBJECTS:
            ix = np.flatnonzero(subject_ids == sid)
            nodes = select_node_features(dataset, 'without_csd', ix).astype(float)
            edges = np.asarray(wpli[ix, :, alpha], dtype=float)
            yy = labels[ix]
            representations = {
                'pooled_9': np.column_stack([nodes.mean(axis=1), edges.mean(axis=1)]),
                'nodes_232': nodes.reshape(len(ix), -1),
                'edges_812': edges,
                'edges_unique_406': edges[:, unique_pair_mask],
                'combined_1044': np.column_stack([nodes.reshape(len(ix), -1), edges]),
            }
            splits = list(StratifiedKFold(5, shuffle=True, random_state=42).split(ix, yy))
            for name, xx in representations.items():
                predictions = np.empty_like(yy)
                for tr, te in splits:
                    steps = [StandardScaler()] if NODE_NORMALIZATION == 'zscore' else []
                    clf = make_pipeline(*steps, LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))
                    clf.fit(xx[tr], yy[tr])
                    predictions[te] = clf.predict(xx[te])
                scores.append({'subject': sid, 'representation': name,
                               'balanced_accuracy': balanced_accuracy_score(yy, predictions)})
    comparison = pd.DataFrame(scores).pivot(index='subject', columns='representation', values='balanced_accuracy')
    comparison['combined_minus_nodes'] = comparison['combined_1044'] - comparison['nodes_232']
    display(comparison)
else:
    print('SKIPPED: set RUN_CLASSIFIERS=True to run the optional comparison.')


## 12. What might still be missing, logically?

| Question | Evidence here | What remains |
|---|---|---|
| Are the saved pairs complete and numerically valid? | Full validator, explicit pair checks, all-trial/band summaries | Valid values can still be scientifically wrong |
| Does matrix extraction preserve pair and band identity? | Known-answer test of the production extractor | Upstream estimator and saving path are not independently verified by that test |
| Are the weights attached to the correct real electrodes/trials? | Metadata order, saved record consistency, loader equality | Independently trace raw EDF channel names, event windows and preprocessing; recompute selected trials and compare by named pair/band |
| Does the loader deliver the intended GCN inputs? | One fold checked against saved arrays, batching and forward pass | Other variants/splits need their own checks |
| Do edges carry predictive information without node power? | Optional edges-only LDA | Run across subjects, assess uncertainty and label permutations |
| Do edges add information beyond nodes? | Optional combined-minus-nodes comparison | Paired evaluation with matched splits; correlated features are not statistically independent merely because inputs were separated |
| Does correct weight placement help the GNN? | Not established here | Retrain matched GNN conditions: true weights, all-one weights, and pair-shuffled weights, using the same splits/seeds |
| Is complete topology the right modeling choice? | Every electrode already connects to every other | A scientific modeling question, not a missing-array bug; compare predeclared alternatives fairly |

For a **pair-shuffle control**, shuffle the 406 unique weights and mirror the result
into reverse edges. Keep node features, labels, topology, and each trial's weight
histogram fixed. State whether the same permutation is used for every trial or an
independent permutation per trial: these test different hypotheses. Repeat across
seeds and retrain each condition. Inference-only scrambling measures sensitivity
to a distribution shift, not the benefit of learning with correct weights.

Permuting `edge_index` columns and weights together changes only storage order.
Permuting all node identities and endpoints together is a relabeling, not destruction
of graph structure. This model also flattens node embeddings in electrode order,
so arbitrary node relabeling is not generally prediction-invariant at its classifier
head. An edges-only vector classifier never consumes `edge_index` at all.

**What we can conclude after the default cells pass:** the current saved data satisfy
the existing structural/numerical contract, the production extractor passes a small
known-answer example, and the selected loader fold preserves the intended data.
We cannot conclude that graph generation is scientifically correct end to end,
that connectivity predicts the label, or that a GNN outperforms simpler models.

Code to inspect next:
- [Generator and preprocessing](../datautils/graphdataversiontwo/generate_dataset.py)
- [Connectivity and matrix extraction](../datautils/graphdataversiontwo/edge_features.py)
- [Full saved-data validator](../datautils/graphdataversiontwo/validation.py)
- [Experiment loader](data/without_csd_alpha_wpli.py)
- [Within-subject training caller](training/within_subject.py)
- [GCN model](model/gcn.py)
